# Forelesning 2: Datahåndtering med Pandas (løsningsforslag)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

## Del 1: Gode kjemiske figurer

In [ ]:
temperature_C = [16, 18, 20, 22, 24]
vapor_pressure_kPa = [1.817, 2.063, 2.339, 2.644, 2.984]

plt.scatter(temperature_C, vapor_pressure_kPa, label="Measurements")
plt.xlabel("Temperature (°C)")
plt.ylabel("Vapor pressure (kPa)")
plt.title("Vapor pressure of water")
plt.legend()
plt.show()

### Skal punktene bindes sammen?

In [ ]:
time_s = [0, 20, 40, 60, 80, 100]
absorbance_298_K = [0.812, 0.552, 0.378, 0.260, 0.181, 0.128]
absorbance_308_K = [0.809, 0.476, 0.284, 0.172, 0.108, 0.071]

plt.plot(time_s, absorbance_298_K, marker="o", label="298 K")
plt.plot(time_s, absorbance_308_K, marker="s", label="308 K")
plt.xlabel("Time (s)")
plt.ylabel("Absorbance")
plt.legend()
plt.show()

**Diskuter:** Absorbansen avtar raskest ved 308 K, så den blandingen reagerer raskest. Figuren alene viser ikke at temperaturen er årsaken. Det forutsetter at temperaturen er den eneste forskjellen mellom forsøkene, og det må forsøksbeskrivelsen fortelle oss.

In [ ]:
time_h = [0, 1, 2, 3, 4]
pH = [7.02, 6.81, 6.60, 6.42, 6.25]

plt.plot(time_h, pH, marker="o", color="tab:blue")
plt.xlabel("Time (h)")
plt.ylabel("pH")
plt.title("pH of the solution over time")
plt.show()

Forbedringer: tid på x-aksen og pH på y-aksen, navn og enhet på begge akser, en farge som synes, markører på målepunktene og en faglig tittel.

## Del 2: Datafiler

**Diskuter:** `"2 µmol/L"` blir lest som tekst, så Pandas kan ikke regne med verdien. Enheten hører hjemme i kolonnenavnet, slik at cellene bare inneholder tall.

## Del 3: Lese og undersøke data

In [ ]:
raw_data = pd.read_csv("../datahandtering_visualisering/data/uvvis_raw.csv")
raw_data

### Hva har vi egentlig lest?

In [ ]:
print(raw_data.shape)
print(raw_data.columns)
raw_data.info()

**Diskuter:** Konsentrasjonen til den ukjente prøven er ikke en feil. Det er den vi skal bestemme. Den andre manglende verdien er absorbansen til `std_6_3`. Det er en manglende måling som må undersøkes.

### Kolonner

In [ ]:
absorbance = raw_data["absorbance"]
print(absorbance.mean())
print(absorbance.max())
raw_data["sample_type"].unique()

In [ ]:
semicolon_data = pd.read_csv("../datahandtering_visualisering/data/example_semicolon.csv", sep=";", decimal=",")
print(semicolon_data)
print(semicolon_data.dtypes)

Uten `sep` og `decimal` deler Pandas linjene ved kommaene. Desimalkommaet blir da tolket som skilletegn, så vi får feil kolonner og tallene 25, 31 og 28 i stedet for 1,25, 1,31 og 1,28. Med `sep=";"` og `decimal=","` får vi to kolonner, og konsentrasjonen blir `float64`.

## Del 4: Rydde data

### Manglende verdier

In [ ]:
print(raw_data.isna().sum())

raw_data[raw_data["absorbance"].isna()]

In [ ]:
print(raw_data.dropna())

clean_data = raw_data.copy()
clean_data = clean_data.dropna(subset=["absorbance"])
print(clean_data.shape)

`dropna()` uten argumenter fjerner også de tre radene med den ukjente prøven, fordi konsentrasjonen mangler der. Med `subset=["absorbance"]` fjernes bare `std_6_3`.

## Del 5: Filtrere, sortere og gruppere

In [ ]:
print(clean_data["sample_type"] == "standard")

standards = clean_data[clean_data["sample_type"] == "standard"]
standards

In [ ]:
blanks = clean_data[clean_data["sample_type"] == "blank"]

low_standards = clean_data[
    (clean_data["sample_type"] == "standard")
    & (clean_data["concentration_uM"] <= 6)
]

known_samples = clean_data[clean_data["sample_type"] != "unknown"]

print(blanks)
print(low_standards)
print(known_samples)

### Sortere

In [ ]:
clean_data.sort_values(by=["sample_type", "concentration_uM", "replicate"])

### Gruppere og oppsummere

In [ ]:
summary = standards.groupby("concentration_uM")["absorbance"].agg(["count", "mean", "std"])
summary

In [ ]:
summary = standards.groupby("concentration_uM")["absorbance"].agg(["count", "mean", "std", "min", "max"])
summary

**Diskuter:** Standardavviket måler spredningen rundt gjennomsnittet, og med én måling finnes det ingen spredning å beregne. Pandas gir da `NaN`. Med bare to eller tre replikater per konsentrasjon er det ikke grunnlag for å si at spredningen øker med konsentrasjonen.

## Del 6: Nye kolonner

In [ ]:
blank_mean = clean_data.loc[clean_data["sample_type"] == "blank", "absorbance"].mean()
print(blank_mean)

clean_data["blank_corrected_absorbance"] = clean_data["absorbance"] - blank_mean
clean_data

In [ ]:
corrected_blanks = clean_data.loc[clean_data["sample_type"] == "blank", "blank_corrected_absorbance"]
print(corrected_blanks.mean())

## Del 7: Fra dataramme til figur

In [ ]:
standards = clean_data[clean_data["sample_type"] == "standard"]

plt.scatter(standards["concentration_uM"], standards["blank_corrected_absorbance"])
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Blank-corrected absorbance")
plt.title("UV-Vis calibration standards")
plt.show()

In [ ]:
plt.errorbar(summary.index, summary["mean"], yerr=summary["std"], fmt="o", capsize=4)
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Absorbance")
plt.show()

**Diskuter:** Med bare gjennomsnitt ser vi ikke enkeltmålingene, hvor mange det er, eller om én av dem skiller seg ut. Feilstolper kan vise standardavvik, standardfeil eller konfidensintervall, og de har svært ulik lengde. Figurteksten må derfor si hva stolpene viser og hvor mange målinger de bygger på.

## Del 8: Rydde tekst og finne duplikater

In [ ]:
messy_data = pd.DataFrame({
    " Sample ID": ["std_2_1", "std_2_2", "std_2_2", "unknown_1"],
    "Sample Type ": ["Standard", "standard ", "standard ", "UNKNOWN"],
    "Absorbance": [0.171, 0.169, 0.169, 0.603],
})

tidy_data = messy_data.copy()
tidy_data.columns = tidy_data.columns.str.strip().str.lower().str.replace(" ", "_")
tidy_data["sample_type"] = tidy_data["sample_type"].str.strip().str.lower()

print(tidy_data["sample_type"].unique())
tidy_data[tidy_data.duplicated(subset=["sample_id"], keep=False)]

**Diskuter:** To rader er en duplikat når de beskriver den samme registreringen, for eksempel samme `sample_id`. To ulike replikater kan godt ha samme absorbans. Da er begge ekte målinger og skal beholdes.

## En reproduserbar arbeidsflyt

In [ ]:
clean_data.to_csv("uvvis_processed.csv", index=False)

## Oppgaver

### Oppgave 1

In [ ]:
kinetics = pd.read_csv("../datahandtering_visualisering/data/reaction_kinetics.csv")
print(kinetics.dtypes)

kinetics = kinetics.sort_values(by="time_s")
kinetics["relative_absorbance"] = kinetics["absorbance"] / kinetics["absorbance"].iloc[0]

plt.plot(kinetics["time_s"], kinetics["relative_absorbance"], marker="o")
plt.xlabel("Time (s)")
plt.ylabel("Relative absorbance")
plt.show()

Absorbansen avtar raskt i starten og langsommere seinere. Dersom absorbansen er proporsjonal med konsentrasjonen, betyr det at stoffet forbrukes, og at reaksjonsfarten avtar når konsentrasjonen blir lavere.

### Oppgave 2

In [ ]:
flame = pd.read_csv("../datahandtering_visualisering/data/flame_emission.csv")
flame.info()

flame.groupby("ion")[["emission_589_nm", "emission_671_nm", "emission_766_nm"]].mean()

Na emitterer sterkest ved 589 nm (gul), Li ved 671 nm (rød) og K ved 766 nm (fiolett flamme, med en sterk linje i det røde og nær-infrarøde). Det stemmer med flammefargene.

### Oppgave 3

In [ ]:
markers = ["o", "s", "^"]

for replicate in [1, 2, 3]:
    series = standards[standards["replicate"] == replicate]
    plt.scatter(series["concentration_uM"], series["absorbance"],
                marker=markers[replicate - 1], label=f"Replicate {replicate}")

plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Absorbance")
plt.legend()
plt.show()

Figuren viser at det ikke er noen systematisk forskjell mellom replikatseriene. Siden punktene nesten ligger oppå hverandre, blir figuren ikke særlig mer informativ, men den kunne avslørt om én serie lå systematisk høyere.

### Oppgave 4

In [ ]:
days = [1, 2, 3, 4, 5]
lab_A = [5.01, 5.03, 5.02, 5.04, 5.03]
lab_B = [5.03, 5.05, 5.04, 5.06, 5.05]

plt.plot(days, lab_A, marker="o", label="Lab A")
plt.plot(days, lab_B, marker="s", label="Lab B")
plt.ylim(5.00, 5.07)
plt.xlabel("Day")
plt.ylabel("Concentration (mg/L)")
plt.title("Misleading")
plt.legend()
plt.show()

plt.plot(days, lab_A, marker="o", label="Lab A")
plt.plot(days, lab_B, marker="s", label="Lab B")
plt.ylim(0, 6)
plt.xlabel("Day")
plt.ylabel("Concentration (mg/L)")
plt.title("Honest")
plt.legend()
plt.show()

Med en smal y-akse ser forskjellen på 0,02 mg/L stor ut. Med aksen fra 0 ser vi at forskjellen er under 1 %. Hva som er redelig, avhenger av om en forskjell på 0,02 mg/L har kjemisk betydning, for eksempel sammenliknet med måleusikkerheten.

### Oppgave 5

In [ ]:
raw_data = pd.read_csv("../datahandtering_visualisering/data/uvvis_raw.csv")
print(raw_data.isna().sum())

clean_data = raw_data.copy()
clean_data = clean_data.dropna(subset=["absorbance"])   # std_6_3 mislyktes ifølge instrumentloggen

blank_mean = clean_data.loc[clean_data["sample_type"] == "blank", "absorbance"].mean()
clean_data["blank_corrected_absorbance"] = clean_data["absorbance"] - blank_mean

clean_data.to_csv("uvvis_processed.csv", index=False)